# 第2章 Vision and Language

# 2.3 LLaVA

## 2.3.3 Transformers で日本語対応 LLaVA 系モデルを動かす

このノートブックは、本書の 2.3.3 項に対応しています。

Huggingface Transformers を使って、日本語対応の軽量 LLaVA 系モデル [Onely7/llava-1.5-llm-jp-3.1-1.8b-instruct4](https://huggingface.co/Onely7/llava-1.5-llm-jp-3.1-1.8b-instruct4) を動かし、画像と日本語の質問から応答を生成します。

前半では、猫の画像に対して「猫は何匹いますか？」と質問し、チャットテンプレートの適用、画像前処理、トークン化、応答生成までを順に実行します。後半では、視覚トークン埋め込みが LLM の入力系列に組み込まれ、次トークンが予測されるまでの流れを確認します。

### 0. 実行環境を準備する

In [1]:
# 必要なライブラリをインストールする
%pip install -q torch "transformers==5.9.0" accelerate pillow requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.8 MB/s eta 0:00:00


### 1. 画像と質問文から応答文を生成する


In [2]:
import requests
import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

model_id = "Onely7/llava-1.5-llm-jp-3.1-1.8b-instruct4"

# モデルを読み込む
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
).to("cuda")
model.eval()

# プロセッサーを読み込む
# プロセッサーは、テキストを処理するトークナイザと、
# 画像を処理する画像前処理器をまとめたものです
processor = AutoProcessor.from_pretrained(
    model_id,
    backend="torchvision"
)

# 会話形式の入力を定義する
# role が "user" の発話に、画像と質問文を配置します
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "猫は何匹いますか？"}
        ]
    }
]

# 会話形式の入力にチャットテンプレートを適用し、
# モデルが期待する形式のプロンプトを作成する
prompt = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True
)

# チャットテンプレート適用後のプロンプトを確認する
print("=== チャットテンプレート適用後のプロンプト ===")
print(prompt)
print()

# 入力画像を読み込む
image_file = "http://images.cocodataset.org/val2017/000000039769.jpg"
raw_image = Image.open(requests.get(image_file, stream=True).raw).convert("RGB")

# 画像とテキストをモデルに入力できるテンソルへ変換する
inputs = processor(
    images=raw_image,
    text=prompt,
    return_tensors="pt"
).to("cuda")

# モデルに渡される入力テンソルの種類と形状を確認する
print("=== モデルに渡される入力テンソル ===")
for key, value in inputs.items():
    print(f"{key}: {value.shape}")
print()

# 推論時には勾配計算を行わず、出力トークン ID を生成する
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

# 生成結果には入力プロンプト部分も含まれるため、
# 新しく生成された応答部分だけを取り出す
generated_answer_ids = generated_ids[:, inputs.input_ids.shape[1] :]

# トークン ID をテキストに変換する
generated_texts = processor.batch_decode(
    generated_answer_ids,
    skip_special_tokens=True
)

print("=== 生成された応答 ===")
print(generated_texts[0])

config.json:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.35GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/614 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/713 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.41M [00:00<?, ?B/s]

=== チャットテンプレート適用後のプロンプト ===
USER: <image>
猫は何匹いますか？ ASSISTANT:

=== モデルに渡される入力テンソル ===
input_ids: torch.Size([1, 592])
attention_mask: torch.Size([1, 592])
pixel_values: torch.Size([1, 3, 336, 336])

=== 生成された応答 ===
画像には2匹の猫がいます。


### 2. 画像情報が LLM の入力系列へ組み込まれる流れを確認する


In [3]:
# input_ids に含まれる画像プレースホルダの数を確認する
image_token_id = model.config.image_token_index
image_token_mask = inputs.input_ids == image_token_id
num_image_tokens = image_token_mask.sum().item()

print("=== 画像プレースホルダ ===")
print(f"画像プレースホルダ数: {num_image_tokens}")
print()

# 画像を視覚エンコーダとアダプタに通し、
# LLM の入力埋め込み次元へ変換された表現を取得する
with torch.inference_mode():
    image_outputs = model.get_image_features(
        pixel_values=inputs.pixel_values
    )

    # 1 枚目の画像に対応する視覚トークン埋め込みを取り出す
    visual_token_embeddings = image_outputs.pooler_output[0]

print("=== 視覚トークン埋め込み ===")
print(f"visual_token_embeddings: {visual_token_embeddings.shape}")
print()

# input_ids に対応するトークン埋め込みを取得する
with torch.inference_mode():
    input_embeds = model.get_input_embeddings()(
        inputs.input_ids
    )

# 視覚トークン埋め込みのデバイスとデータ型をそろえる
visual_token_embeddings = visual_token_embeddings.to(
    input_embeds.device,
    input_embeds.dtype
)

# 元の埋め込み系列をコピーする
llm_input_embeds = input_embeds.clone()

# 画像プレースホルダ位置の埋め込みを、
# 視覚トークン埋め込みで置き換える
llm_input_embeds[image_token_mask] = visual_token_embeddings

num_visual_embeddings = visual_token_embeddings.shape[0]

print("=== LLM に入力される埋め込み系列 ===")
print(f"input_embeds: {input_embeds.shape}")
print(f"llm_input_embeds: {llm_input_embeds.shape}")
print(
    "画像プレースホルダ数と視覚トークン埋め込み数が一致:",
    num_image_tokens == num_visual_embeddings
)
print(
    "画像プレースホルダ位置が視覚トークン埋め込みに置き換わった:",
    torch.allclose(
        llm_input_embeds[image_token_mask],
        visual_token_embeddings
    )
)
print()

# 作成した埋め込み系列を LLM に入力し、次トークンを予測する
# 視覚情報はすでに llm_input_embeds に含まれているため、
# ここでは pixel_values をモデルへ渡さない
with torch.inference_mode():
    outputs = model(
        inputs_embeds=llm_input_embeds,
        attention_mask=inputs.attention_mask,
        # 次トークン予測に必要な最後の１位置分の logits だけを算出
        logits_to_keep=1
    )

# 系列の最後の位置における語彙上のスコアを取り出す
# 形状は [バッチサイズ, 語彙数]（この例では [1, 99584]）
next_token_logits = outputs.logits[:, -1, :]

# 最もスコアの高いトークンを次トークンとして選ぶ
next_token_id = next_token_logits.argmax(dim=-1)

print("=== llm_input_embeds からの次トークン予測 ===")
print(f"次トークン ID: {next_token_id.item()}")
print(
    "次トークン:",
    processor.batch_decode(
        next_token_id.unsqueeze(-1),
        skip_special_tokens=False
    )[0]
)

# In[2] の model.generate が最初に生成したトークンと比較する
generated_first_token_id = generated_answer_ids[:, 0]

print(
    f"model.generate の最初のトークン ID: "
    f"{generated_first_token_id.item()}"
)
print(
    "model.generate の最初のトークンと一致:",
    torch.equal(
        next_token_id,
        generated_first_token_id
    )
)

=== 画像プレースホルダ ===
画像プレースホルダ数: 576

=== 視覚トークン埋め込み ===
visual_token_embeddings: torch.Size([576, 2048])

=== LLM に入力される埋め込み系列 ===
input_embeds: torch.Size([1, 592, 2048])
llm_input_embeds: torch.Size([1, 592, 2048])
画像プレースホルダ数と視覚トークン埋め込み数が一致: True
画像プレースホルダ位置が視覚トークン埋め込みに置き換わった: True

=== llm_input_embeds からの次トークン予測 ===
次トークン ID: 51666
次トークン: 画像
model.generate の最初のトークン ID: 51666
model.generate の最初のトークンと一致: True
